# Explainable AI (XAI) with SHAP

**Description:** This notebook explores **Explainable AI (XAI)** using the **SHAP (SHapley Additive exPlanations)** library. Machine learning models, especially complex ones like Random Forests or Neural Networks, are often "black boxes." SHAP values help us understand *why* a model made a specific prediction by quantifying the contribution of each feature.

We will cover:
1.  **Synthetic Data Generation**: Creating a realistic "Credit Risk" dataset.
2.  **Model Training**: Training a Random Forest Classifier to predict default risk.
3.  **SHAP Explainer**: calculating SHAP values.
4.  **Global Interpretability**: Summary plots to see overall feature importance.
5.  **Local Interpretability**: Force plots to explain individual predictions.
6.  **Dependence Plots**: Understanding interaction effects between features.

# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q shap scikit-learn pandas numpy matplotlib
    print('Packages installed.')

In [ ]:
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Ensure reproducibility
np.random.seed(42)

## 1. Generate Synthetic Credit Risk Data
We'll simulate a dataset representing loan applicants. 
Features:
*   `Income`: Annual income (in thousands).
*   `Debt`: Total debt amount.
*   `Age`: Applicant's age.
*   `YearsEmployed`: Years at current job.
*   `CreditScore`: Synthetic credit score.

Target:
*   `Default`: 1 if they defaulted, 0 otherwise. We'll make it dependent on Debt and Income.

In [ ]:
n_samples = 1000

# Generate features
income = np.random.normal(50, 15, n_samples)  # Mean 50k, SD 15k
debt = np.random.normal(10, 5, n_samples)     # Mean 10k, SD 5k
age = np.random.randint(20, 70, n_samples)
years_employed = np.random.randint(0, 20, n_samples)
credit_score = np.random.randint(300, 850, n_samples)

# Generate target (Default) with some logic + noise
# Higher debt, lower income, lower credit score -> higher chance of default
risk_score = (debt * 2) - (income * 0.5) - ((credit_score - 600) * 0.1) + np.random.normal(0, 10, n_samples)
default = (risk_score > 0).astype(int)

df = pd.DataFrame({
    'Income': income,
    'Debt': debt,
    'Age': age,
    'YearsEmployed': years_employed,
    'CreditScore': credit_score,
    'Default': default
})

df.head()

## 2. Train a Random Forest Classifier
We use a Random Forest, which is a powerful but non-linear model. Interpreting it is harder than a Logistic Regression, making it a perfect candidate for SHAP.

In [ ]:
X = df.drop('Default', axis=1)
y = df['Default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model Accuracy on Test Set:", model.score(X_test, y_test))

## 3. Calculate SHAP Values
The `TreeExplainer` is optimized for tree-based models like Random Forest, XGBoost, and LightGBM.

In [ ]:
# Initialize JavaScript visualization for the notebook
shap.initjs()

# Create explainer
explainer = shap.TreeExplainer(model)

# Calculate SHAP values for the test set
shap_values = explainer.shap_values(X_test)

# Handle different SHAP versions/outputs
if isinstance(shap_values, list):
    # Old SHAP behavior: list of arrays [class0, class1]
    shap_values_class1 = shap_values[1]
else:
    # New SHAP behavior: 3D array (samples, features, classes)
    # We want class 1 (index 1 in the last dimension)
    shap_values_class1 = shap_values[:, :, 1]

## 4. Global Interpretability: Summary Plot
The summary plot combines feature importance with feature effects.
*   **Y-axis**: Features ordered by importance.
*   **X-axis**: SHAP value (impact on model output).
*   **Color**: Feature value (Red = High, Blue = Low).

**Interpretation**:
*   If high values of a feature (Red) are on the right (positive SHAP), that feature increases the risk of default.
*   If high values are on the left (negative SHAP), that feature decreases the risk.

In [ ]:
plt.title("SHAP Summary Plot (Class 1: Default)")
shap.summary_plot(shap_values_class1, X_test)

## 5. Local Interpretability: Force Plot
We can explain *single* predictions. Why did the model predict that Applicant #0 would (or would not) default?

*   **Red bars**: Push the prediction higher (towards Default).
*   **Blue bars**: Push the prediction lower (towards Safe).

In [ ]:
# Explain the first instance in the test set
instance_idx = 0
print(f"Actual Label: {y_test.iloc[instance_idx]}")
print(f"Predicted Probability (Default): {model.predict_proba(X_test.iloc[[instance_idx]])[0][1]:.2f}")

# Use matplotlib=True for static rendering in some environments, or default for JS
shap.force_plot(explainer.expected_value[1], shap_values_class1[instance_idx], X_test.iloc[instance_idx], matplotlib=True)

## 6. Dependence Plot
Dependence plots show the relationship between a feature's value and its SHAP value. It helps visualize non-linear relationships and interactions.

Let's look at `Debt`. We expect higher debt to increase risk (positive SHAP).

In [ ]:
shap.dependence_plot("Debt", shap_values_class1, X_test)

## Conclusion
SHAP provides a unified framework to interpret model predictions. By understanding which features drive the model's decisions (e.g., Debt and Income being primary drivers), social scientists can validate if the model aligns with theoretical expectations and identify potential biases.